# SITS - Merge de Dados

Este notebook combina multiplos arquivos NPZ em um unico dataset para treinamento.

**Casos de uso:**
- Combinar anotacoes manuais (notebook 01) com dados externos (notebook 02)
- Unificar dados de diferentes fontes/projetos
- Definir ordem personalizada das classes

**Saida:**
- `dataset.npz`: Dataset unificado (X, y)
- `dataset_metadata.json`: Mapeamento de classes e estatisticas

> **Nota:** Se voce tem apenas um arquivo NPZ, pode pular este notebook.

## 1. Configuracao

In [ ]:
from pathlib import Path
import json
import numpy as np

# =============================================================================
# CONFIGURACAO - EDITE AQUI
# =============================================================================

# Pasta com os arquivos NPZ
DATA_FOLDER = Path("../sessions/seu_projeto/annotation")

# Arquivos de entrada (adicione quantos precisar)
INPUT_FILES = [
    {
        "npz": DATA_FOLDER / "anotacoes.npz",
        "metadata": DATA_FOLDER / "anotacoes_metadata.json",
        "name": "Anotacoes",  # Nome para exibicao
    },
    {
        "npz": DATA_FOLDER / "dados_externos.npz",
        "metadata": DATA_FOLDER / "dados_externos_metadata.json",
        "name": "Externos",
    },
]

# Arquivos de saida
OUTPUT_NPZ = DATA_FOLDER / "dataset.npz"
OUTPUT_META = DATA_FOLDER / "dataset_metadata.json"

# Nomes das bandas (para visualizacao)
BAND_NAMES = ["blue", "green", "red", "nir"]

# =============================================================================
# ORDEM PERSONALIZADA DAS CLASSES (opcional)
# =============================================================================
# Deixe None para ordem alfabetica, ou defina a ordem desejada
CUSTOM_CLASS_ORDER = None
# Exemplo:
# CUSTOM_CLASS_ORDER = [
#     "agua",
#     "floresta",
#     "agricultura",
#     "urbano",
# ]

# =============================================================================

print(f"Pasta de dados: {DATA_FOLDER}")
print(f"Arquivos de entrada: {len(INPUT_FILES)}")
for f in INPUT_FILES:
    print(f"  - {f['name']}: {f['npz'].name}")

## 2. Verificar Arquivos

In [ ]:
print("Verificando arquivos de entrada:")
print("=" * 60)

files_ok = True
for f in INPUT_FILES:
    print(f"\n{f['name']}:")
    
    # NPZ
    if f["npz"].exists():
        size = f["npz"].stat().st_size / 1024
        print(f"  [OK] {f['npz'].name} ({size:.1f} KB)")
    else:
        print(f"  [!!] {f['npz'].name} NAO ENCONTRADO")
        files_ok = False
    
    # Metadata
    if f["metadata"].exists():
        size = f["metadata"].stat().st_size / 1024
        print(f"  [OK] {f['metadata'].name} ({size:.1f} KB)")
    else:
        print(f"  [!!] {f['metadata'].name} NAO ENCONTRADO")
        files_ok = False

if not files_ok:
    print("\nERRO: Alguns arquivos nao foram encontrados.")
    print("Execute os notebooks anteriores primeiro.")
else:
    print("\nTodos os arquivos encontrados!")

## 3. Carregar Dados

In [ ]:
datasets = []

for f in INPUT_FILES:
    print(f"\n{f['name'].upper()}")
    print("-" * 40)
    
    # Carregar NPZ
    data = np.load(f["npz"])
    X = data["X"]
    y_idx = data["y"]
    
    # Carregar metadata
    with open(f["metadata"], "r", encoding="utf-8") as file:
        meta = json.load(file)
    
    # Obter mapeamento idx -> nome
    if "idx_to_name" in meta:
        idx_to_name = {int(k): v for k, v in meta["idx_to_name"].items()}
    elif "class_mapping" in meta:
        idx_to_name = {v: k for k, v in meta["class_mapping"].items()}
    else:
        # Fallback: usar indices como nomes
        idx_to_name = {i: f"classe_{i}" for i in range(int(y_idx.max()) + 1)}
    
    print(f"  Shape: {X.shape}")
    print(f"  Classes:")
    for idx in sorted(idx_to_name.keys()):
        name = idx_to_name[idx]
        count = np.sum(y_idx == idx)
        print(f"    {idx}: {name} - {count} amostras")
    
    # Converter indices para nomes
    y_names = np.array([idx_to_name[idx] for idx in y_idx])
    
    datasets.append({
        "name": f["name"],
        "X": X,
        "y_names": y_names,
        "idx_to_name": idx_to_name,
    })

print(f"\n\nTotal: {len(datasets)} datasets carregados")

## 4. Verificar Compatibilidade

In [ ]:
print("Verificacao de compatibilidade:")
print("=" * 60)

# Verificar shapes (C, T)
shapes = [(d["name"], d["X"].shape[1:]) for d in datasets]
unique_shapes = set(s[1] for s in shapes)

print("\nShapes (C, T):")
for name, shape in shapes:
    print(f"  {name}: {shape}")

if len(unique_shapes) == 1:
    print("\n[OK] Todos os shapes sao compativeis!")
else:
    print("\n[ERRO] Shapes incompativeis!")
    print("Todos os datasets precisam ter o mesmo numero de canais e timesteps.")

# Estatisticas
print("\nEstatisticas:")
for d in datasets:
    X = d["X"]
    print(f"  {d['name']:20s} - min: {X.min():.1f}, max: {X.max():.1f}, mean: {X.mean():.1f}")

## 5. Merge

In [ ]:
# Concatenar todos os datasets
X = np.concatenate([d["X"] for d in datasets], axis=0)
y_names = np.concatenate([d["y_names"] for d in datasets], axis=0)

# Descobrir todas as classes unicas
unique_classes = sorted(set(y_names))

# Definir ordem das classes
if CUSTOM_CLASS_ORDER is not None:
    # Usar ordem personalizada
    missing = set(unique_classes) - set(CUSTOM_CLASS_ORDER)
    if missing:
        print(f"AVISO: Classes nao definidas na ordem: {missing}")
        print("Adicionando ao final...")
        CUSTOM_CLASS_ORDER = list(CUSTOM_CLASS_ORDER) + sorted(missing)
    
    class_order = [c for c in CUSTOM_CLASS_ORDER if c in unique_classes]
    order_type = "personalizada"
else:
    # Ordem alfabetica
    class_order = unique_classes
    order_type = "alfabetica"

# Criar mapeamento
class_mapping = {name: idx for idx, name in enumerate(class_order)}
idx_to_name = {idx: name for name, idx in class_mapping.items()}

# Converter nomes para indices
y = np.array([class_mapping[name] for name in y_names], dtype=np.int64)

print("DATASET MERGED")
print("=" * 60)
print(f"X shape: {X.shape}  (N, C, T)")
print(f"y shape: {y.shape}  (N,)")
print(f"Ordem das classes: {order_type}")
print()
print(f"Classes: {len(class_mapping)}")
print("-" * 60)
for idx in range(len(idx_to_name)):
    name = idx_to_name[idx]
    count = np.sum(y == idx)
    pct = count / len(y) * 100
    print(f"  {idx:2d}: {name:25s} - {count:6d} ({pct:5.1f}%)")
print("-" * 60)
print(f"      {'TOTAL':25s} - {len(y):6d}")

## 6. Visualizacao

In [ ]:
import matplotlib.pyplot as plt

# Graficos de distribuicao
classes = [idx_to_name[i] for i in range(len(idx_to_name))]
counts = [np.sum(y == i) for i in range(len(idx_to_name))]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Barras
colors = plt.cm.tab10(np.linspace(0, 1, len(classes)))
bars = axes[0].bar(classes, counts, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_xlabel("Classe")
axes[0].set_ylabel("Quantidade")
axes[0].set_title("Distribuicao do Dataset")
axes[0].tick_params(axis='x', rotation=45)
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(counts)*0.01,
                 str(count), ha='center', va='bottom', fontsize=9)

# Pizza
axes[1].pie(counts, labels=classes, colors=colors, autopct='%1.1f%%',
            startangle=90, explode=[0.02] * len(classes))
axes[1].set_title("Proporcao por Classe")

plt.tight_layout()
plt.show()

In [ ]:
# Exemplos de series temporais
n_classes = len(class_mapping)
n_cols = min(4, n_classes)
n_rows = (n_classes + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(4*n_cols, 3*n_rows))
if n_classes == 1:
    axes = np.array([[axes]])
axes = np.atleast_2d(axes).flatten()

for idx in range(n_classes):
    ax = axes[idx]
    name = idx_to_name[idx]
    
    # Primeira amostra da classe
    sample_indices = np.where(y == idx)[0]
    if len(sample_indices) == 0:
        continue
    sample = X[sample_indices[0]]
    
    for b, band in enumerate(BAND_NAMES):
        ax.plot(sample[b], label=band, linewidth=1.5)
    
    ax.set_title(name)
    ax.set_xlabel("Timestep")
    ax.legend(loc='upper right', fontsize=8)
    ax.grid(True, alpha=0.3)

# Esconder eixos vazios
for idx in range(n_classes, len(axes)):
    axes[idx].axis('off')

plt.suptitle("Exemplo por classe", fontsize=14)
plt.tight_layout()
plt.show()

## 7. Salvar

In [ ]:
from datetime import datetime

# Criar pasta se nao existir
OUTPUT_NPZ.parent.mkdir(parents=True, exist_ok=True)

# Salvar NPZ
np.savez_compressed(OUTPUT_NPZ, X=X, y=y)

# Salvar metadados
metadata = {
    "class_mapping": class_mapping,
    "idx_to_name": {str(k): v for k, v in idx_to_name.items()},
    "n_samples": int(X.shape[0]),
    "n_channels": int(X.shape[1]),
    "n_timesteps": int(X.shape[2]),
    "n_classes": len(class_mapping),
    "band_names": BAND_NAMES,
    "class_order": order_type,
    "sources": [f["npz"].name for f in INPUT_FILES],
    "statistics": {idx_to_name[idx]: int(np.sum(y == idx)) for idx in range(len(idx_to_name))},
    "created": datetime.now().isoformat(),
}

with open(OUTPUT_META, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

# Mostrar resultado
npz_size = OUTPUT_NPZ.stat().st_size / 1024 / 1024
json_size = OUTPUT_META.stat().st_size / 1024

print("Arquivos salvos:")
print(f"  {OUTPUT_NPZ.name}: {npz_size:.2f} MB")
print(f"  {OUTPUT_META.name}: {json_size:.1f} KB")
print()
print(f"Caminho: {OUTPUT_NPZ.parent.absolute()}")

In [ ]:
# Verificar integridade
data_check = np.load(OUTPUT_NPZ)
assert np.allclose(X, data_check["X"]), "X nao corresponde!"
assert np.array_equal(y, data_check["y"]), "y nao corresponde!"
print("Verificacao de integridade: OK")

---

## Resumo

Dataset unificado criado!

**Arquivos gerados:**
- `dataset.npz`: Arrays X (timeseries) e y (labels)
- `dataset_metadata.json`: Mapeamento de classes e estatisticas

**Proximo passo:** Execute **04_treinamento.ipynb** para treinar modelos.